In [ ]:
"""
=============================================================
   HFT FOOTPRINT + UT BOT HYBRID STRATEGY
=============================================================

STRATEGY 1:
    HFT Footprint Score

STRATEGY 2:
    UT Bot Trend Signals

STRATEGY 3:
    Combined HFT + UT Bot Confirmation

BACKTEST OUTPUT:
    ✔ Final Capital
    ✔ Win Rate
    ✔ Sharpe Ratio
    ✔ Max Drawdown
    ✔ Equity Curve
    ✔ Side-by-side comparison

DATA:
    Binance BTCUSDT 1m candles

=============================================================
"""

import ccxt
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# =========================================================
# SETTINGS
# =========================================================
SYMBOL = "BTC/USDT"
TIMEFRAME = "1m"

INITIAL_CAPITAL = 100000

RISK_PER_TRADE = 0.10

STOP_LOSS_PCT = 0.015
TAKE_PROFIT_PCT = 0.03

HFT_THRESHOLD = 4

UT_ATR_PERIOD = 14

LIMIT = 5000

# =========================================================
# DOWNLOAD DATA
# =========================================================
exchange = ccxt.binance()

print("Downloading data...")

ohlcv = exchange.fetch_ohlcv(
    SYMBOL,
    timeframe=TIMEFRAME,
    limit=LIMIT
)

df = pd.DataFrame(
    ohlcv,
    columns=[
        "timestamp",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
)

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    unit="ms"
)

# =========================================================
# HFT FOOTPRINT ENGINE
# =========================================================

# ---------------------------------------------------------
# 1. Volume Spike
# ---------------------------------------------------------
df["vol_ma"] = (
    df["Volume"]
    .rolling(20)
    .mean()
)

df["volume_spike"] = (
    df["Volume"]
    >
    df["vol_ma"] * 2
).astype(int)

# ---------------------------------------------------------
# 2. Spread Compression
# ---------------------------------------------------------
df["spread"] = (
    (df["High"] - df["Low"])
    /
    df["Close"]
)

df["spread_ma"] = (
    df["spread"]
    .rolling(20)
    .mean()
)

df["spread_compression"] = (
    df["spread"]
    <
    df["spread_ma"] * 0.7
).astype(int)

# ---------------------------------------------------------
# 3. Momentum Burst
# ---------------------------------------------------------
df["return"] = df["Close"].pct_change()

df["momentum_burst"] = (
    abs(df["return"])
    >
    df["return"]
    .rolling(20)
    .std()
    * 2
).astype(int)

# ---------------------------------------------------------
# 4. Liquidity Absorption
# ---------------------------------------------------------
df["absorption"] = (
    (
        (df["Volume"] > df["vol_ma"] * 1.5)
        &
        (
            abs(df["return"])
            <
            df["return"]
            .rolling(20)
            .std()
        )
    )
).astype(int)

# ---------------------------------------------------------
# 5. Trend Pressure
# ---------------------------------------------------------
df["ema_fast"] = (
    df["Close"]
    .ewm(span=10)
    .mean()
)

df["ema_slow"] = (
    df["Close"]
    .ewm(span=30)
    .mean()
)

df["trend_pressure"] = (
    df["ema_fast"]
    >
    df["ema_slow"]
).astype(int)

# =========================================================
# HFT SCORE
# =========================================================
df["hft_score"] = (
    df["volume_spike"]
    +
    df["spread_compression"]
    +
    df["momentum_burst"]
    +
    df["absorption"]
    +
    df["trend_pressure"]
)

# =========================================================
# UT BOT
# =========================================================
tr = np.maximum(
    df["High"] - df["Low"],
    np.maximum(
        abs(df["High"] - df["Close"].shift()),
        abs(df["Low"] - df["Close"].shift())
    )
)

df["atr"] = tr.rolling(UT_ATR_PERIOD).mean()

df["upper"] = (
    df["Close"]
    - df["atr"]
)

df["lower"] = (
    df["Close"]
    + df["atr"]
)

trend = [1]

for i in range(1, len(df)):

    if df["Close"].iloc[i] > df["lower"].iloc[i - 1]:
        trend.append(1)

    elif df["Close"].iloc[i] < df["upper"].iloc[i - 1]:
        trend.append(-1)

    else:
        trend.append(trend[-1])

df["trend"] = trend

df["ut_buy"] = (
    (df["trend"] == 1)
    &
    (df["trend"].shift() == -1)
)

df["ut_sell"] = (
    (df["trend"] == -1)
    &
    (df["trend"].shift() == 1)
)

# =========================================================
# SIGNALS
# =========================================================

# HFT ONLY
df["hft_buy"] = (
    df["hft_score"]
    >= HFT_THRESHOLD
)

# COMBINED SIGNAL
df["combined_buy"] = (
    df["hft_buy"]
    &
    df["ut_buy"]
)

# =========================================================
# BACKTEST FUNCTION
# =========================================================
def backtest(signal_column):

    capital = INITIAL_CAPITAL

    equity_curve = []

    trade_log = []

    position = None

    for i in range(50, len(df)):

        row = df.iloc[i]

        price = row["Close"]

        # ENTRY
        if (
            position is None
            and row[signal_column]
        ):

            allocation = capital * RISK_PER_TRADE

            shares = allocation / price

            position = {

                "entry_price": price,
                "shares": shares,
                "invested": allocation
            }

            capital -= allocation

        # EXIT
        elif position is not None:

            entry = position["entry_price"]

            stop_price = (
                entry
                * (1 - STOP_LOSS_PCT)
            )

            tp_price = (
                entry
                * (1 + TAKE_PROFIT_PCT)
            )

            if (
                price <= stop_price
                or price >= tp_price
                or row["ut_sell"]
            ):

                proceeds = (
                    position["shares"]
                    * price
                )

                profit = (
                    proceeds
                    - position["invested"]
                )

                capital += proceeds

                trade_log.append(profit)

                position = None

        # EQUITY
        equity = capital

        if position is not None:

            equity += (
                position["shares"]
                * price
            )

        equity_curve.append(equity)

    # =====================================================
    # METRICS
    # =====================================================
    equity_series = pd.Series(equity_curve)

    daily_returns = (
        equity_series
        .pct_change()
        .dropna()
    )

    sharpe = 0

    if daily_returns.std() != 0:

        sharpe = (
            daily_returns.mean()
            /
            daily_returns.std()
        ) * np.sqrt(365)

    wins = [x for x in trade_log if x > 0]

    win_rate = 0

    if len(trade_log) > 0:

        win_rate = (
            len(wins)
            /
            len(trade_log)
        ) * 100

    rolling_max = equity_series.cummax()

    drawdown = (
        equity_series
        - rolling_max
    ) / rolling_max

    max_dd = drawdown.min() * 100

    return {
        "Final Capital": round(equity_curve[-1], 2),
        "Trades": len(trade_log),
        "Win Rate": round(win_rate, 2),
        "Sharpe": round(sharpe, 2),
        "Max Drawdown": round(max_dd, 2),
        "Equity Curve": equity_curve
    }

# =========================================================
# RUN STRATEGIES
# =========================================================
print("\nRunning HFT strategy...")
hft_results = backtest("hft_buy")

print("Running UT Bot strategy...")
ut_results = backtest("ut_buy")

print("Running Combined strategy...")
combined_results = backtest("combined_buy")

# =========================================================
# COMPARISON
# =========================================================
comparison = pd.DataFrame({

    "HFT": [
        hft_results["Final Capital"],
        hft_results["Trades"],
        hft_results["Win Rate"],
        hft_results["Sharpe"],
        hft_results["Max Drawdown"]
    ],

    "UT BOT": [
        ut_results["Final Capital"],
        ut_results["Trades"],
        ut_results["Win Rate"],
        ut_results["Sharpe"],
        ut_results["Max Drawdown"]
    ],

    "COMBINED": [
        combined_results["Final Capital"],
        combined_results["Trades"],
        combined_results["Win Rate"],
        combined_results["Sharpe"],
        combined_results["Max Drawdown"]
    ]

},

index=[
    "Final Capital",
    "Trades",
    "Win Rate %",
    "Sharpe Ratio",
    "Max Drawdown %"
])

print("\n================================================")
print(comparison)
print("================================================")

# =========================================================
# PLOT EQUITY CURVES
# =========================================================
plt.figure(figsize=(14, 7))

plt.plot(
    hft_results["Equity Curve"],
    label="HFT"
)

plt.plot(
    ut_results["Equity Curve"],
    label="UT BOT"
)

plt.plot(
    combined_results["Equity Curve"],
    label="COMBINED"
)

plt.title(
    "HFT vs UT BOT vs COMBINED"
)

plt.xlabel("Trades")

plt.ylabel("Portfolio Value")

plt.legend()

plt.grid(True)

plt.show()

# =========================================================
# NORMAL DISTRIBUTION OF RETURNS
# =========================================================
returns = df["return"].dropna()

mu = returns.mean()
sigma = returns.std()

x = np.linspace(
    returns.min(),
    returns.max(),
    100
)

y = (
    1
    /
    (
        sigma
        * np.sqrt(2 * np.pi)
    )
) * np.exp(
    - (
        (x - mu) ** 2
    ) / (
        2 * sigma ** 2
    )
)

plt.figure(figsize=(12, 6))

plt.hist(
    returns,
    bins=50,
    density=True,
    alpha=0.6
)

plt.plot(x, y)

plt.title(
    "Normal Distribution of Returns"
)

plt.xlabel("Returns")

plt.ylabel("Density")

plt.grid(True)

plt.show()

print("\nDone.")